# Crypto market regime detection: probabilistic validation

BTC changes character over time. Some weeks are quiet and directional. Some are noisy rallies. Some look like stress that has not finished clearing.

I use a Gaussian Mixture Model to label those periods from recent BTC/USDT and ETH/USDT daily candles. The goal is description, not price prediction. The useful question is whether the labels survive train/test separation, baselines, and stability checks.


In [ ]:
from datetime import timezone
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
BASE_URL = "https://api.binance.com/api/v3/klines"
START_DATE = "2023-01-01"
INTERVAL = "1d"
SYMBOLS = ["BTCUSDT", "ETHUSDT"]

PROJECT_DIR = Path.cwd() if Path.cwd().name == "crypto-market-regime-detection" else Path("notebooks/crypto-market-regime-detection")
ASSET_DIR = PROJECT_DIR / "assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["font.family"] = "DejaVu Sans"


## 1. Pull recent market data

The data comes from Binance public candles. Daily bars are enough for this version, and no API key is needed.


In [ ]:
def fetch_klines(symbol, start_date=START_DATE, interval=INTERVAL):
    start = pd.Timestamp(start_date, tz="UTC")
    end = pd.Timestamp.now(tz="UTC").normalize() + pd.Timedelta(days=1)
    rows = []
    cursor = int(start.timestamp() * 1000)
    end_ms = int(end.timestamp() * 1000)

    while cursor < end_ms:
        response = requests.get(
            BASE_URL,
            params={"symbol": symbol, "interval": interval, "startTime": cursor, "limit": 1000},
            timeout=30,
        )
        response.raise_for_status()
        batch = response.json()
        if not batch:
            break
        rows.extend(batch)
        next_cursor = int(batch[-1][0]) + 24 * 60 * 60 * 1000
        if next_cursor <= cursor:
            break
        cursor = next_cursor

    columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore",
    ]
    out = pd.DataFrame(rows, columns=columns)
    numeric_cols = ["open", "high", "low", "close", "volume", "quote_asset_volume", "number_of_trades"]
    out[numeric_cols] = out[numeric_cols].apply(pd.to_numeric, errors="coerce")
    out["date"] = pd.to_datetime(out["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    out["symbol"] = symbol
    return out[["date", "symbol", "open", "high", "low", "close", "volume", "quote_asset_volume", "number_of_trades"]]

raw = pd.concat([fetch_klines(symbol) for symbol in SYMBOLS], ignore_index=True)
raw = raw.drop_duplicates(["symbol", "date"]).sort_values(["symbol", "date"]).reset_index(drop=True)
raw.groupby("symbol").agg(start=("date", "min"), end=("date", "max"), rows=("date", "size"), last_close=("close", "last"))


## 2. Feature engineering

The features are the usual market-regime suspects: return, trend, realized volatility, downside volatility, drawdown, range, volume shock, trade intensity, and ETH/BTC relative strength.


In [ ]:
def add_features(group):
    group = group.sort_values("date").copy()
    group["log_return"] = np.log(group["close"]).diff()
    group["range_pct"] = np.log(group["high"] / group["low"])
    group["trend_14"] = np.log(group["close"] / group["close"].shift(14))
    group["trend_30"] = np.log(group["close"] / group["close"].shift(30))
    group["vol_7"] = group["log_return"].rolling(7).std() * np.sqrt(365)
    group["vol_30"] = group["log_return"].rolling(30).std() * np.sqrt(365)
    downside = group["log_return"].where(group["log_return"] < 0, 0)
    group["downside_vol_30"] = downside.rolling(30).std() * np.sqrt(365)
    group["drawdown_90"] = group["close"] / group["close"].rolling(90).max() - 1
    group["volume_z_30"] = (np.log1p(group["volume"]) - np.log1p(group["volume"]).rolling(30).mean()) / np.log1p(group["volume"]).rolling(30).std()
    group["trade_intensity_z_30"] = (np.log1p(group["number_of_trades"]) - np.log1p(group["number_of_trades"]).rolling(30).mean()) / np.log1p(group["number_of_trades"]).rolling(30).std()
    return group

features_by_symbol = raw.groupby("symbol", group_keys=False).apply(add_features)
wide_close = raw.pivot(index="date", columns="symbol", values="close").sort_index()
wide_return = np.log(wide_close).diff()
relative = (wide_return["ETHUSDT"] - wide_return["BTCUSDT"]).rolling(7).sum().rename("eth_btc_relative_7")

btc = features_by_symbol[features_by_symbol["symbol"].eq("BTCUSDT")].copy()
btc = btc.merge(relative.reset_index(), on="date", how="left")

MODEL_FEATURES = [
    "log_return",
    "trend_14",
    "trend_30",
    "vol_7",
    "vol_30",
    "downside_vol_30",
    "drawdown_90",
    "range_pct",
    "volume_z_30",
    "trade_intensity_z_30",
    "eth_btc_relative_7",
]

model_df = btc.dropna(subset=MODEL_FEATURES).sort_values("date").reset_index(drop=True)
model_df["split"] = np.where(np.arange(len(model_df)) < int(len(model_df) * 0.8), "train", "test")
model_df[["date", "close", "split"] + MODEL_FEATURES].head()


## 3. Model selection and temporal validation

I fit GMMs with 2 to 6 regimes on the training window. BIC chooses the number of regimes there, then the held-out tail gets used as a sanity check.

I also compare two plain baselines:

- KMeans on the same standardized features
- volatility buckets from 30-day realized volatility


In [ ]:
train_mask = model_df["split"].eq("train").to_numpy()
test_mask = ~train_mask

scaler = StandardScaler()
X_train = scaler.fit_transform(model_df.loc[train_mask, MODEL_FEATURES])
X_test = scaler.transform(model_df.loc[test_mask, MODEL_FEATURES])
X_all = scaler.transform(model_df[MODEL_FEATURES])

selection_rows = []
models = {}
for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, covariance_type="full", n_init=20, random_state=RANDOM_STATE)
    gmm.fit(X_train)
    models[k] = gmm
    train_labels = gmm.predict(X_train)
    test_labels = gmm.predict(X_test)
    selection_rows.append(
        {
            "k": k,
            "bic_train": gmm.bic(X_train),
            "aic_train": gmm.aic(X_train),
            "train_log_likelihood": gmm.score(X_train),
            "test_log_likelihood": gmm.score(X_test),
            "train_silhouette": silhouette_score(X_train, train_labels),
            "test_silhouette": silhouette_score(X_test, test_labels),
        }
    )

selection = pd.DataFrame(selection_rows)
selected_k = int(selection.loc[selection["bic_train"].idxmin(), "k"])
gmm = models[selected_k]

model_df["gmm_component"] = gmm.predict(X_all)
model_df["assignment_probability"] = gmm.predict_proba(X_all).max(axis=1)
model_df["gmm_train_selected"] = selected_k

kmeans = KMeans(n_clusters=selected_k, n_init=50, random_state=RANDOM_STATE)
kmeans.fit(X_train)
model_df["kmeans_component"] = kmeans.predict(X_all)

vol_bins = pd.qcut(model_df["vol_30"], q=selected_k, labels=False, duplicates="drop")
model_df["volatility_regime"] = vol_bins.astype(int)

selection


## 4. Regime naming, risk, and persistence

The model gives component numbers. I turn those into analyst labels only after looking at return, volatility, drawdown, tail risk, and duration.


In [ ]:
def describe_regimes(data, label_col):
    rows = []
    for label, group in data.groupby(label_col):
        r = group["log_return"].dropna()
        var_95 = float(np.percentile(r, 5))
        es_95 = float(r[r <= var_95].mean()) if (r <= var_95).any() else var_95
        rows.append(
            {
                "component": int(label),
                "days": int(len(group)),
                "share": len(group) / len(data),
                "avg_daily_return": float(r.mean()),
                "ann_return_approx": float(r.mean() * 365),
                "ann_vol": float(r.std(ddof=1) * np.sqrt(365)),
                "var_95_daily": var_95,
                "expected_shortfall_95_daily": es_95,
                "avg_drawdown_90": float(group["drawdown_90"].mean()),
                "median_assignment_probability": float(group["assignment_probability"].median()) if "assignment_probability" in group else np.nan,
                "trend_30": float(group["trend_30"].mean()),
                "volume_z_30": float(group["volume_z_30"].mean()),
            }
        )
    return pd.DataFrame(rows)

regime_stats = describe_regimes(model_df, "gmm_component")

def regime_name(row):
    if row["ann_vol"] >= regime_stats["ann_vol"].quantile(0.75) and row["avg_drawdown_90"] < regime_stats["avg_drawdown_90"].median():
        return "Stress / drawdown"
    if row["ann_return_approx"] > 0.45 and row["ann_vol"] >= regime_stats["ann_vol"].median():
        return "Volatile rally"
    if row["ann_return_approx"] > 0.20 and row["ann_vol"] < regime_stats["ann_vol"].median():
        return "Calm uptrend"
    if abs(row["ann_return_approx"]) <= 0.20 and row["ann_vol"] < regime_stats["ann_vol"].median():
        return "Sideways / low momentum"
    if row["avg_drawdown_90"] < -0.10:
        return "Risk-off consolidation"
    return "Mixed transition"

regime_stats["regime"] = regime_stats.apply(regime_name, axis=1)
name_map = dict(zip(regime_stats["component"], regime_stats["regime"]))
model_df["regime"] = model_df["gmm_component"].map(name_map)

# Carry the analyst names back into the summary table.
regime_stats = regime_stats.sort_values("ann_return_approx", ascending=False).reset_index(drop=True)

ordered = model_df.sort_values("date").copy()
ordered["next_regime"] = ordered["regime"].shift(-1)
transition = pd.crosstab(ordered["regime"], ordered["next_regime"], normalize="index").fillna(0)
transition_counts = pd.crosstab(ordered["regime"], ordered["next_regime"]).fillna(0)

# Length of consecutive runs in each regime.
runs = []
run_id = (ordered["regime"] != ordered["regime"].shift()).cumsum()
for _, group in ordered.groupby(run_id):
    runs.append({"regime": group["regime"].iloc[0], "duration_days": len(group)})
durations = pd.DataFrame(runs)
duration_stats = durations.groupby("regime").agg(avg_duration_days=("duration_days", "mean"), max_duration_days=("duration_days", "max")).reset_index()
regime_stats = regime_stats.merge(duration_stats, on="regime", how="left")

regime_stats


## 5. Stability and baseline comparison

A regime model can look convincing from one fit. Bootstrap stability is the cold shower: refit the same number of regimes on resampled training windows, predict the whole series, and compare labels with adjusted Rand index.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
ari_scores = []
base_labels = model_df["gmm_component"].to_numpy()
for i in range(40):
    sample_idx = rng.choice(np.where(train_mask)[0], size=train_mask.sum(), replace=True)
    boot = GaussianMixture(n_components=selected_k, covariance_type="full", n_init=10, random_state=RANDOM_STATE + i + 1)
    boot.fit(X_all[sample_idx])
    boot_labels = boot.predict(X_all)
    ari_scores.append(adjusted_rand_score(base_labels, boot_labels))

baseline_rows = []
for label_col, label_name in [
    ("gmm_component", "GMM selected by BIC"),
    ("kmeans_component", "KMeans same k"),
    ("volatility_regime", "Volatility quantile baseline"),
]:
    labels = model_df[label_col].to_numpy()
    same_next = np.mean(labels[1:] == labels[:-1])
    baseline_rows.append(
        {
            "model": label_name,
            "n_regimes": int(pd.Series(labels).nunique()),
            "silhouette_all": float(silhouette_score(X_all, labels)) if pd.Series(labels).nunique() > 1 else np.nan,
            "one_day_persistence": float(same_next),
        }
    )

baseline_comparison = pd.DataFrame(baseline_rows)
stability_summary = {
    "bootstrap_ari_median": float(np.median(ari_scores)),
    "bootstrap_ari_p10": float(np.percentile(ari_scores, 10)),
    "bootstrap_ari_p90": float(np.percentile(ari_scores, 90)),
}

baseline_comparison, stability_summary


## 6. Report charts

These charts are the public-facing report: timeline, model selection, trend-volatility map, transition matrix, risk table, confidence, and baseline comparison.


In [ ]:
regime_order = regime_stats.sort_values("ann_vol", ascending=False)["regime"].tolist()
palette_values = ["#B91C1C", "#D97706", "#1B6CA8", "#047857", "#6D28D9", "#64748B"]
palette = {regime: palette_values[i % len(palette_values)] for i, regime in enumerate(regime_order)}

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5), sharex=True, gridspec_kw={"height_ratios": [2.4, 1]})
plot_df = model_df.sort_values("date")
axes[0].plot(plot_df["date"], plot_df["close"], color="#111827", lw=1.4)
for regime, group in plot_df.groupby("regime"):
    axes[0].scatter(group["date"], group["close"], s=14, color=palette[regime], label=regime, alpha=0.75)
axes[0].set_yscale("log")
axes[0].set_title("BTC price colored by probabilistic market regime")
axes[0].set_ylabel("BTCUSDT close, log scale")
axes[0].legend(ncol=2, frameon=False, fontsize=9, loc="upper left")
axes[0].axvline(plot_df.loc[test_mask, "date"].min(), color="#475569", ls="--", lw=1)
axes[0].text(plot_df.loc[test_mask, "date"].min(), axes[0].get_ylim()[0] * 1.08, "test window", rotation=90, fontsize=9, color="#475569")

axes[1].plot(plot_df["date"], plot_df["assignment_probability"], color="#1B6CA8", lw=1)
axes[1].axhline(0.8, color="#64748B", ls="--", lw=1)
axes[1].set_ylabel("Assignment probability")
axes[1].set_xlabel("")
axes[1].set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(ASSET_DIR / "01_btc_regime_timeline.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
axes[0].plot(selection["k"], selection["bic_train"], marker="o", color="#1B6CA8", label="BIC")
axes[0].plot(selection["k"], selection["aic_train"], marker="o", color="#C46A2B", label="AIC")
axes[0].axvline(selected_k, color="#111827", ls="--", lw=1)
axes[0].set_title("Regime count selection on train window")
axes[0].set_xlabel("Number of regimes")
axes[0].set_ylabel("Information criterion")
axes[0].legend(frameon=False)

axes[1].plot(selection["k"], selection["train_log_likelihood"], marker="o", color="#1B6CA8", label="Train")
axes[1].plot(selection["k"], selection["test_log_likelihood"], marker="o", color="#047857", label="Test")
axes[1].axvline(selected_k, color="#111827", ls="--", lw=1)
axes[1].set_title("Out-of-sample log likelihood")
axes[1].set_xlabel("Number of regimes")
axes[1].set_ylabel("Average log likelihood")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(ASSET_DIR / "00_model_selection_validation.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 7))
sns.scatterplot(
    data=model_df,
    x="vol_30",
    y="trend_30",
    hue="regime",
    size="assignment_probability",
    sizes=(25, 120),
    palette=palette,
    alpha=0.75,
    ax=ax,
)
ax.axhline(0, color="#111827", lw=1)
ax.set_title("Regimes in trend-volatility space")
ax.set_xlabel("30-day realized volatility, annualized")
ax.set_ylabel("30-day log trend")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(ASSET_DIR / "02_return_volatility_map.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
transition_plot = transition.reindex(index=regime_order, columns=regime_order).fillna(0)
sns.heatmap(transition_plot, annot=True, fmt=".0%", cmap="Blues", cbar=False, ax=ax)
ax.set_title("One-day regime transition matrix")
ax.set_xlabel("Next day regime")
ax.set_ylabel("Current regime")
fig.tight_layout()
fig.savefig(ASSET_DIR / "03_transition_matrix.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
risk_table = regime_stats.copy()
risk_table["Ann. return"] = risk_table["ann_return_approx"].map(lambda x: f"{x:.1%}")
risk_table["Ann. vol"] = risk_table["ann_vol"].map(lambda x: f"{x:.1%}")
risk_table["VaR 95"] = risk_table["var_95_daily"].map(lambda x: f"{x:.1%}")
risk_table["ES 95"] = risk_table["expected_shortfall_95_daily"].map(lambda x: f"{x:.1%}")
risk_table["Median p"] = risk_table["median_assignment_probability"].map(lambda x: f"{x:.0%}")
risk_table["Avg duration"] = risk_table["avg_duration_days"].map(lambda x: f"{x:.1f}d")
display_cols = ["regime", "days", "Ann. return", "Ann. vol", "VaR 95", "ES 95", "Avg duration", "Median p"]
ax.axis("off")
table = ax.table(cellText=risk_table[display_cols].values, colLabels=display_cols, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#CBD5E1")
    if row == 0:
        cell.set_facecolor("#E2E8F0")
        cell.set_text_props(weight="bold")
ax.set_title("Risk profile by regime", pad=18)
fig.tight_layout()
fig.savefig(ASSET_DIR / "04_regime_risk_table.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
sns.histplot(model_df["assignment_probability"], bins=24, color="#1B6CA8", ax=axes[0])
axes[0].axvline(model_df["assignment_probability"].median(), color="#111827", ls="--", lw=1)
axes[0].set_title("GMM assignment confidence")
axes[0].set_xlabel("Maximum posterior probability")
axes[0].set_ylabel("Days")

sns.histplot(ari_scores, bins=16, color="#047857", ax=axes[1])
axes[1].axvline(np.median(ari_scores), color="#111827", ls="--", lw=1)
axes[1].set_title("Bootstrap stability of regime assignments")
axes[1].set_xlabel("Adjusted Rand index vs selected model")
axes[1].set_ylabel("Bootstrap refits")
fig.tight_layout()
fig.savefig(ASSET_DIR / "05_assignment_confidence.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.8))
base = baseline_comparison.sort_values("silhouette_all", ascending=True)
ax.barh(base["model"], base["silhouette_all"], color=["#1B6CA8" if "GMM" in m else "#94A3B8" for m in base["model"]])
ax.set_title("Baseline comparison in standardized feature space")
ax.set_xlabel("Silhouette score")
ax.set_ylabel("")
for i, row in enumerate(base.itertuples()):
    ax.text(row.silhouette_all + 0.005, i, f"persistence {row.one_day_persistence:.0%}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(ASSET_DIR / "06_baseline_comparison.png", bbox_inches="tight")
plt.show()


## 7. Readout

The last cell writes the run summary to JSON so the README can quote exact numbers from the notebook run.


In [ ]:
latest = model_df.sort_values("date").iloc[-1]
test_df = model_df.loc[test_mask].copy()
selected_row = selection[selection["k"].eq(selected_k)].iloc[0]

summary = {
    "data_start": model_df["date"].min().strftime("%Y-%m-%d"),
    "data_end": model_df["date"].max().strftime("%Y-%m-%d"),
    "btc_rows": int((raw["symbol"].eq("BTCUSDT")).sum()),
    "model_rows": int(len(model_df)),
    "train_rows": int(train_mask.sum()),
    "test_rows": int(test_mask.sum()),
    "selected_regimes": selected_k,
    "train_bic_selected": float(selected_row["bic_train"]),
    "test_log_likelihood_selected": float(selected_row["test_log_likelihood"]),
    "latest_regime": str(latest["regime"]),
    "latest_regime_probability": float(latest["assignment_probability"]),
    "latest_btc_close": float(latest["close"]),
    "median_assignment_probability": float(model_df["assignment_probability"].median()),
    "tail_median_assignment_probability": float(test_df["assignment_probability"].median()),
    "bootstrap_ari_median": stability_summary["bootstrap_ari_median"],
    "bootstrap_ari_p10": stability_summary["bootstrap_ari_p10"],
    "bootstrap_ari_p90": stability_summary["bootstrap_ari_p90"],
    "gmm_silhouette": float(baseline_comparison.loc[baseline_comparison["model"].eq("GMM selected by BIC"), "silhouette_all"].iloc[0]),
    "gmm_one_day_persistence": float(baseline_comparison.loc[baseline_comparison["model"].eq("GMM selected by BIC"), "one_day_persistence"].iloc[0]),
}

(PROJECT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
